In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import os

RUNS_DIR = '../runs'
RUNS = {
    'Baseline': 'ppo_baseline_seed42',
    'No Reward Norm (A1)': 'ppo_no-reward-norm_seed42',
    'No Frame Stack (A2)': 'ppo_no-frame-stack_seed42',
    'No Orth Init (A3)': 'ppo_no-orth-init_seed42',
    'Constant LR (A4)': 'ppo_no-lr-decay_seed42',
}

def load_scalar(run_dir, tag):
    ea = EventAccumulator(run_dir)
    ea.Reload()
    if tag not in ea.Tags()['scalars']:
        return None, None
    events = ea.Scalars(tag)
    steps = [e.step for e in events]
    values = [e.value for e in events]
    return np.array(steps), np.array(values)

def smooth(values, window=50):
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode='valid')

print('Setup OK')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for (label, run_name), color in zip(RUNS.items(), colors):
    run_dir = os.path.join(RUNS_DIR, run_name)
    steps, values = load_scalar(run_dir, 'charts/episodic_return')
    if steps is None:
        print(f'No data for {label}')
        continue
    smoothed = smooth(values, window=100)
    steps_s = steps[99:]
    ax.plot(steps_s / 1e6, smoothed, label=label, color=color, linewidth=2)

ax.set_xlabel('Environment Steps (M)', fontsize=12)
ax.set_ylabel('Episodic Return (smoothed)', fontsize=12)
ax.set_title('Training Dynamics: Episodic Return — Baseline vs Ablations', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('../report/figures', exist_ok=True)
plt.savefig('../report/figures/reward_curves.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for (label, run_name), color in zip(RUNS.items(), colors):
    run_dir = os.path.join(RUNS_DIR, run_name)
    steps, values = load_scalar(run_dir, 'charts/episodic_return')
    if steps is None:
        continue
    smoothed = smooth(values, window=100)
    steps_s = steps[99:]
    ax.plot(steps_s / 1e6, smoothed, label=label, color=color, linewidth=2)

ax.set_xlabel('Environment Steps (M)', fontsize=12)
ax.set_ylabel('Episodic Return (smoothed)', fontsize=12)
ax.set_title('Training Dynamics: Episodic Return — Baseline vs Ablations', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('../report/figures', exist_ok=True)
plt.savefig('../report/figures/reward_curves.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for (label, run_name), color in zip(RUNS.items(), colors):
    run_dir = os.path.join(RUNS_DIR, run_name)
    steps, values = load_scalar(run_dir, 'losses/entropy')
    if steps is None:
        continue
    ax.plot(steps / 1e6, values, label=label, color=color, linewidth=1.5)

ax.set_xlabel('Environment Steps (M)', fontsize=12)
ax.set_ylabel('Policy Entropy', fontsize=12)
ax.set_title('Policy Entropy Over Training', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../report/figures/entropy.pdf', bbox_inches='tight')
plt.show()

In [ ]:
results = {}
for label, run_name in RUNS.items():
    run_dir = os.path.join(RUNS_DIR, run_name)
    steps, values = load_scalar(run_dir, 'charts/episodic_return')
    if values is None:
        results[label] = {'Final Return (mean)': 'N/A', 'Final Return (std)': 'N/A'}
        continue
    final_values = values[-200:] if len(values) >= 200 else values
    results[label] = {
        'Final Return (mean)': f'{np.mean(final_values):.1f}',
        'Final Return (std)': f'+/-{np.std(final_values):.1f}',
    }

df = pd.DataFrame(results).T
print(df.to_string())
df.to_csv('../report/figures/ablation_table.csv')